### Generic library and functions import

In [8]:
# Import packages
from matplotlib.ticker import MultipleLocator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib as mpl
import math
import os
import h5py
import ast
import openpyxl
import pickle
from gams import *
import gams.transfer as gt
idx = pd.IndexSlice
import matplotlib.patches as patches
from matplotlib.patches import Patch
from adjustText import adjust_text
import seaborn as sns
import json
import geopandas as gpd
import folium
from folium import plugins


#import skimage as ski
from matplotlib.font_manager import FontProperties

from openpyxl import load_workbook


#_OUTPUT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))) + '/output'
_EPM_DIR = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))) + '\\EPM'
_RESOURCES_DIR = os.path.join(_EPM_DIR, 'epm/resources')
_INPUT_DIR = os.path.join(_EPM_DIR, 'epm/input')

#Extracting resources
#COLORS
COLORS = os.path.join(_RESOURCES_DIR, 'postprocess/colors.csv')
colors = pd.read_csv(COLORS, skiprows=1)
colors = colors.dropna(subset=['Processing'])
colors['Processing'] = colors['Processing'].astype(str).str.strip()
colors = colors[~colors['Processing'].str.startswith('#')]
colors = colors.dropna(subset=['Color'])
colors['Color'] = colors['Color'].astype(str).str.strip()
colors = colors[colors['Color'] != '']
base_colors = colors.set_index('Processing')['Color'].dropna().to_dict()

def PlotProp(ax, label_size, tick_type = ""):
    ax.grid(linestyle='--', linewidth=0.5, axis='y')
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize= label_size)
    ax.tick_params(axis='both', which='minor', labelsize= label_size)    
    if tick_type == 'years':
        ax.xaxis.set_major_locator(MultipleLocator(5))
        ax.xaxis.set_minor_locator(MultipleLocator(1))

def ax_change(ax):
    ax.yaxis.set_tick_params(labelleft=True)
    for t in ax.get_yticklabels():
        t.set_ha('left')
        t.set_va('bottom')

def ax_properties(ax, label_size = 11):
    PlotProp(ax, label_size)
    ax_change(ax)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', which='major', labelsize=label_size)

def bar_labels(ax, containers_labels = True, container_lim = 0.5, container_round = 1, container_fontsize =7, container_color = 'white', container_rot = 0 ,
                    top_labels = True, totals = [], total_alttype = False ,text_unit = None, y_offset = 0.02, total_round = 1, totals_fontsize = 8, totals_color = 'black', totals_rot = 0 ):
    if containers_labels:
        for c in ax.containers:
            # Optional: if the segment is small or 0, customize the labels
            if container_round >0:
                labels = [round(v.get_height(),container_round) if abs(v.get_height()) > container_lim else '' for v in c]
            else:
                labels = [round(v.get_height()) if abs(v.get_height()) > container_lim else '' for v in c]

            # remove the labels parameter if it's not needed for customized labels
            ax.bar_label(c, labels=labels, label_type='edge', fontweight = 'bold', fontsize = container_fontsize, rotation = container_rot, padding = 0.2, color = container_color)

    if top_labels:
        if total_alttype:
            # Set an offset that is used to bump the label up a bit above the bar.
            y_offset = y_offset*ax.get_ylim()[1]
            # Add labels to each bar.
            for i, total in enumerate(totals):
                if text_unit is not None:
                    txt = str(round(total,total_round)) + ' ' + text_unit
                else:
                    txt = round(total,total_round)
                ax.text(i, ax.get_ylim()[1] + y_offset,txt , ha='center',
                    weight='bold', fontsize = totals_fontsize, rotation = totals_rot, color = totals_color)
        else:
            # Set an offset that is used to bump the label up a bit above the bar.
            y_offset = y_offset*ax.get_ylim()[1]
            # Add labels to each bar.
            for i, total in enumerate(totals):
                if text_unit is not None:
                    txt = str(round(total,total_round)) + ' ' + text_unit
                else:
                    txt = round(total,total_round)
                ax.text(i, total + y_offset,txt , ha='center',
                    weight='bold', fontsize = totals_fontsize, rotation = totals_rot, color = totals_color)
                
def plot_separators(ax, df):

    myticklabels= df.index.get_level_values(level=1).tolist()
    ## First level labels as per subject 
    ax.set_xticklabels(myticklabels, rotation=0)

    ## Draw vertical lines (I am not including the lines at the ends)
    pos=[]
    pos.append(-0.5)

    level_0_length = len(df.index.unique(level=0))
    len_1_unique = len(df.index.unique(level=1))
    for i in range(level_0_length):
        pos.append((i+1)*len_1_unique-0.5)
    ax.vlines(pos[1:-1], 0, -0.2, color='black', lw=1.5, clip_on=False, transform=ax.get_xaxis_transform())
    ax.vlines(pos[1:-1], 0, ax.get_ylim()[1], color='black', lw=0.5, linestyle='--',clip_on=True, transform=ax.get_xaxis_transform())
    ## Draw second level axes ticklables
    for ps0, ps1, lbl in zip(pos[:-1], pos[1:], df.index.unique(level=0)):
        ax.text((ps0 + ps1) / 2, -0.1, lbl, ha='center', clip_on=False, transform=ax.get_xaxis_transform(), weight = 'bold',rotation=0, size=13)


colours = ["#2E45B8", "#1DC9A4", "#E2365B", "#FB9851",'#595959', "#B3B3B3", "#B4F1AD", "#AFA76D", "#D0E1E1", "#8055ED","#C30ED7D2","#FFCE54","#21CFE6","#5D0303E7","#4FC1E9","#AC92EC","#EC87C0","#494a4b","#CCD1D9","#656D78"]
stylelines = ['--', '-.', '-', ':']
markerlines = ['.', ',', 'o', 'v', '^', '<', '>']
econ_colors = ['#E3120B', '#1F2E7A', '#475ED1','#1DC9A4']
fuel_colours = {'Gas':"#4FC1E9", 'LNG':"#AC92EC", 'Water':"#2E45B8", 'PV':"#FFCE54", 'HFO':'red', 'Battery':"#E1DFD0", 'Diesel':"#656D78",
                  'Wind':"#1DC9A4", 'Rooftop':"#FB9851", 'Offshore':"#A0D468", 'Biomas':"#ED5565", 'Coal':"#494a4b", 'GasCCS': "#FC6E51"}
years = ['2025','2026','2027','2028','2029','2030','2031','2032','2033','2034','2035', '2038', '2040', '2042', '2045', '2048', '2050']
pWeightYears = dict(zip(years, [1,2,3,2,3,3,2,2,3,3,2,3,2,3,3,2,3]))

### Reading the data from the output directory and storing it in a pickle file for further analysis

In [9]:
#Specify the output directory and filename for storing the processed results
output_dir = os.path.join(_EPM_DIR, 'epm\\output\\latest')
output_filename = f'output_{os.path.basename(output_dir)}.pkl'

if not os.path.exists(output_filename):
    data = {}
    ## Specifying folder for output processing
    ## Retrieving scenarios
    scenarios_list = [f for f in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, f)) and f != 'img' and not f.startswith('CO2') and not f.endswith('CoalPhaseOut')]
    print("Scenarios found: ", scenarios_list)
    for j in scenarios_list:
        try:
            dir_main =  "%s/epmresults.gdx"%(os.path.join(output_dir, j))
            m_main = gt.Container(dir_main)
            data[j] = {}
            data[j]["pEnergyByFuelCountry"] =  m_main.data['pEnergyTechFuel'].records
            data[j]['pDemandPeakZone'] = m_main.data['pDemandPeakZone'].records
            data[j]["pCapexInvestmentComponent"] =  m_main.data['pCapexInvestmentComponent'].records
            data[j]["pCostsSummaryZone"] =  m_main.data['pNetPresentCost'].records
            data[j]["pEmissionsZone"] =  m_main.data['pEmissionsZone'].records
            data[j]["pCostsZone"] =  m_main.data['pCosts'].records
            data[j]["pHourlyPrice"] =  m_main.data['pHourlyPrice'].records
            data[j]["pCostsPlant"] =  m_main.data['pCostsPlant'].records
            data[j]["pCosts"] =  m_main.data['pNetPresentCost'].records
            data[j]["pCapacityByFuelCountry"] =  m_main.data['pCapacityTechFuel'].records
            data[j]['pEnergyPlant'] = m_main.data['pEnergyPlant'].records
            data[j]["pSystemAverageCost"] =  m_main.data['pCostsSystemPerMWh'].records
            data[j]["pZoneAverageCost"] = m_main.data['pGenCostsPerMWh'].records
            data[j]["pStorageComponents"] = m_main.data['pStorageComponents'].records
            data[j]["pUtilizationByFuelCountry"] = m_main.data['pUtilizationTechFuel'].records
            data[j]["pReserveSpinningPlantZone"] = m_main.data['pReserveSpinningPlantZone'].records
            data[j]["pDemandSupplyCountry"] =  m_main.data['pEnergyBalance'].records
            data[j]["pNewCapacityFuelCountry"] =  m_main.data['pNewCapacityTechFuel'].records
            data[j]["pNewTransmissionCapacity"] = m_main.data['pNewTransmissionCapacity'].records
            data[j]["pTransmissionCapacity"] = m_main.data['pTransmissionCapacity'].records
            data[j]["pInterconUtilization"] =  m_main.data['pInterconUtilization'].records
            data[j]["pInterchangeCountry"] =  m_main.data['pInterchange'].records
            data[j]["pEmissionsCountry"] =  m_main.data['pEmissionsZone'].records
            data[j]["pFuelDispatch"] = m_main.data['pDispatchTechFuel'].records
            data[j]["pDispatch"] = m_main.data['pDispatch'].records
            data[j]["pDispatchPlant"] = m_main.data['pDispatchPlant'].records
            data[j]["pSummary"] =  m_main.data['pNetPresentCostSystem'].records
            data[j]["pCountryAverageCost"] =  m_main.data['pCostsPerMWh'].records
            data[j]["pNetImport"] = m_main.data['pNetImport'].records
            data[j]["pInterchange"] = m_main.data['pInterchange'].records
            data[j]["pInterchangeExternalExports"] = m_main.data['pInterchangeExternalExports'].records
            data[j]["pInterchangeExternalImports"] = m_main.data['pInterchangeExternalImports'].records
            data[j]["pSummary"]['Case'] = j
            data[j]['pRR'] = m_main.data['pRR'].records
            data[j]['pZoneCountry'] = m_main.data['pZoneCountry'].records
            print("Data extracted for scenario ", j) 
        
        except Exception as e:
            print(f"Unexpected error for scenario {j}: {e}")
            pass
    print("=================================")
    # Rename scenarios with more intuitive names
    # Scenarios are : ['baseline', 'baseline_INGA', 'baseline_INGAv2', 'baseline_limexp', 'baseline_limTrade', 'baseline_noBTKG', 'baseline_ref']
    # Create the mapping dictionary
"""    scenario_mapping = {
        'baseline': 'Baseline',
        'baseline_INGA': 'Baseline + INGA',
        'baseline_INGAv2': 'Baseline + INGA v2',
        'baseline_limexp': 'Limited VRE Expansion',
        'baseline_limTrade': 'Limited Regional Integration',
        'baseline_noBTKG': 'Baseline + No BTKG',
        'baseline_ref': 'Limited Interconnection Expansion'
    }
    print("Renaming scenarios for better understanding...")
    for old_name, new_name in scenario_mapping.items():
        if old_name in data:
            data[new_name] = data.pop(old_name)
    print("Scenarios output successfully processed: ", data.keys())
    print("Storing results data to file: ", output_filename) """

with open(output_filename, 'wb') as f:
        pickle.dump(data, f)
"""else:
print("Loading data from file: ", output_filename)
with open(output_filename, 'rb') as f:
        data = pickle.load(f)"""
scenarios = data.keys()

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'c:\\My_Stuff\\_ActiveModels\\WB2015_PowerSystems\\GamsProjects\\EPM\\epm\\output\\latest'

### Results exports

In [ ]:
#Extracting files for project assessments and building scenario pairs for comparison
def extract_data_for_assessment(datasource, component):
    for scenario in datasource:
        datasource[scenario][component]['Scenario'] = scenario
    df = pd.concat([datasource[scenario][component] for scenario in datasource], ignore_index=True)

    return df

#Creating dataframes
#Transmission capacity
#df_transmission = extract_data_for_assessment(data, 'pTransmissionCapacity')
#Imports and exports
df_exchanges = extract_data_for_assessment(data, 'pDemandSupplyCountry')
#Generation by fuel
df_generation = extract_data_for_assessment(data, 'pEnergyByFuelCountry')
#Dispatch by fuel
#df_dispatch = extract_data_for_assessment(data, 'pFuelDispatch')
#Interchange by country
df_interchange = extract_data_for_assessment(data, 'pInterchange')
#Interchange Exports by country
#df_interchange_exports = extract_data_for_assessment(data, 'pInterchangeExternalExports')
#Interchange Imports by country
#df_interchange_imports = extract_data_for_assessment(data, 'pInterchangeExternalImports')
# Generation costs
df_gen_costs = extract_data_for_assessment(data, 'pZoneAverageCost')
#CAPEX and transmission investments
df_capex = extract_data_for_assessment(data, 'pCapexInvestmentComponent')
#Spinning and other cost components
df_costs = extract_data_for_assessment(data, 'pCostsZone')
#Capacity by fuel
df_capacity = extract_data_for_assessment(data, 'pCapacityByFuelCountry')
#Emissions
#df_emissions = extract_data_for_assessment(data, 'pEmissionsZone')
#Creating dataframe for generation changes

#Export dataframes to excel for further analysis
with pd.ExcelWriter(os.path.join(_EPM_DIR, 'epm/output/simulations_CAPP/processed_data_bis.xlsx')) as writer:
    #df_transmission.to_excel(writer, sheet_name='TransmissionCapacity', index=False)
    df_exchanges.to_excel(writer, sheet_name='Exchanges', index=False)
    df_generation.to_excel(writer, sheet_name='GenerationByFuel', index=False)
    df_interchange.to_excel(writer, sheet_name='Interchange', index=False)
    #df_interchange_exports.to_excel(writer, sheet_name='InterchangeExports', index=False)
    #df_interchange_imports.to_excel(writer, sheet_name='InterchangeImports', index=False)
    df_capex.to_excel(writer, sheet_name='CapexInvestment', index=False)
    df_gen_costs.to_excel(writer, sheet_name='GenCosts', index=False)
    df_costs.to_excel(writer, sheet_name='CostsZone', index=False)
    df_capacity.to_excel(writer, sheet_name='CapacityByFuel', index=False)
    #df_emissions.to_excel(writer, sheet_name='EmissionsZone', index=False)
   # df_dispatch.to_excel(writer, sheet_name='DispatchByFuel', index=False)